# Name: Aleeha Aiman
# RollNo: 23-NTU-CS-1243
# Class: BsCs 7th(A)


#ACTIVITY 1

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

In [5]:
import kagglehub
import shutil
import os

path = kagglehub.dataset_download("mlg-ulb/creditcardfraud")
shutil.copy(os.path.join(path, "creditcard.csv"), "creditcard.csv")

Using Colab cache for faster access to the 'creditcardfraud' dataset.


'creditcard.csv'

In [6]:
df = pd.read_csv('creditcard.csv')
print(df.shape)
print(df['Class'].value_counts())
print(df['Class'].value_counts(normalize=True) * 100)
df.head()

(284807, 31)
Class
0    284315
1       492
Name: count, dtype: int64
Class
0    99.827251
1     0.172749
Name: proportion, dtype: float64


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [7]:
X = df.drop('Class', axis=1)
y = df['Class']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(X_train.shape, X_test.shape)
print(y_train.value_counts())
print(y_test.value_counts())

(227845, 30) (56962, 30)
Class
0    227451
1       394
Name: count, dtype: int64
Class
0    56864
1       98
Name: count, dtype: int64


In [8]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [9]:
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train_scaled, y_train)
print(pd.Series(y_train_res).value_counts())

Class
0    227451
1    227451
Name: count, dtype: int64


In [10]:
knn_base = KNeighborsClassifier(n_neighbors=5, n_jobs=-1)
knn_base.fit(X_train_scaled, y_train)
y_pred_base = knn_base.predict(X_test_scaled)
acc_base = accuracy_score(y_test, y_pred_base)
f1_base = f1_score(y_test, y_pred_base)
print('Accuracy:', acc_base)
print('F1-Score:', f1_base)
print(classification_report(y_test, y_pred_base, digits=4))

Accuracy: 0.9995435553526912
F1-Score: 0.8586956521739131
              precision    recall  f1-score   support

           0     0.9997    0.9999    0.9998     56864
           1     0.9186    0.8061    0.8587        98

    accuracy                         0.9995     56962
   macro avg     0.9591    0.9030    0.9292     56962
weighted avg     0.9995    0.9995    0.9995     56962



In [ ]:
knn_smote = KNeighborsClassifier(n_neighbors=5, n_jobs=-1)
knn_smote.fit(X_train_res, y_train_res)
y_pred_smote = knn_smote.predict(X_test_scaled)
acc_smote = accuracy_score(y_test, y_pred_smote)
f1_smote = f1_score(y_test, y_pred_smote)
print('Accuracy:', acc_smote)
print('F1-Score:', f1_smote)
print(classification_report(y_test, y_pred_smote, digits=4))

In [ ]:
X_tune, _, y_tune, _ = train_test_split(X_train_scaled, y_train, train_size=60000, random_state=42, stratify=y_train)
pipe = ImbPipeline([
    ('smote', SMOTE(random_state=42)),
    ('knn', KNeighborsClassifier(n_jobs=-1))
])
param_grid = {'knn__n_neighbors': [3, 5, 7, 9, 11]}
grid = GridSearchCV(pipe, param_grid, scoring='f1', cv=3, n_jobs=1, verbose=2)
grid.fit(X_tune, y_tune)
print('Best k:', grid.best_params_['knn__n_neighbors'])
print('Best CV F1:', grid.best_score_)
pd.DataFrame(grid.cv_results_)[['param_knn__n_neighbors', 'mean_test_score', 'std_test_score']]

In [ ]:
best_k = grid.best_params_['knn__n_neighbors']
knn_best = KNeighborsClassifier(n_neighbors=best_k, n_jobs=-1)
knn_best.fit(X_train_res, y_train_res)
y_pred_best = knn_best.predict(X_test_scaled)
acc_best = accuracy_score(y_test, y_pred_best)
f1_best = f1_score(y_test, y_pred_best)
print('Best k:', best_k)
print('Accuracy:', acc_best)
print('F1-Score:', f1_best)
print(classification_report(y_test, y_pred_best, digits=4))

In [ ]:
results = pd.DataFrame({
    'Model': ['KNN k=5 (No SMOTE)', 'KNN k=5 (SMOTE)', f'KNN k={best_k} (SMOTE + GridSearchCV)'],
    'Accuracy': [acc_base, acc_smote, acc_best],
    'F1-Score': [f1_base, f1_smote, f1_best]
})
results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
results_melt = results.melt(id_vars='Model', var_name='Metric', value_name='Score')
sns.barplot(data=results_melt, x='Metric', y='Score', hue='Model', ax=axes[0])
axes[0].set_ylim(0, 1.05)
axes[0].set_title('Model Comparison')
axes[0].legend(fontsize=8)
sns.heatmap(confusion_matrix(y_test, y_pred_best), annot=True, fmt='d', cmap='Blues', ax=axes[1])
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')
axes[1].set_title('Confusion Matrix (Tuned KNN + SMOTE)')
plt.tight_layout()
plt.show()